In [17]:
import os
import re
from kiwipiepy import Kiwi
from dotenv import load_dotenv
from ibm_watsonx_ai.foundation_models import ModelInference
from ibm_watsonx_ai.metanames import GenTextParamsMetaNames as GenParams

load_dotenv()

True

In [18]:
credentials = {
    "url": os.getenv("WATSONX_URL"),
    "apikey": os.getenv("WATSONX_APIKEY")
}
project_id = os.getenv("WATSONX_PROJECT_ID")

In [19]:
db_fetched_data = """
놀이터에서 처음 본 또래한테 아끼는 모래놀이 장난감 선뜻 나눠줌. 언제 이렇게 자랐나 싶어 멀리서 보는데 괜히 마음 뭉클함. 오는 길에 초코 간식 사줌.
"""

In [20]:
kiwi = Kiwi()

tokens = kiwi.tokenize(db_fetched_data)
replacements = []

for i in range(len(tokens) - 1):
    t1 = tokens[i]    # 앞 단어
    t2 = tokens[i+1]  # 뒤에 오는 명사
    
    # 1. 뒤의 단어가 일반명사(NNG)인지 확인
    if t2.tag == "NNG":
        # 원문에서 해당 명사 바로 앞의 1글자짜리 단어 구역을 안전하게 추출
        # 형태소 분석기의 오류를 방지하기 위해 부사(MAG)와 동사/형용사(VV/VA) 활용형을 모두 포괄합니다.
        if t1.tag in ["VV", "VA", "MAG"]:
            original_word = db_fetched_data[t1.start:t1.end].strip()
            
            # 오해를 일으키는 핵심 원인인 '1글자 조사/수식어'인 경우만 저격
            if len(original_word) == 1:
                target_phrase = f"{original_word} {t2.form}"
                
                # 분석기가 찾아낸 원래 품사의 '원형(Lemma)'을 추출 (예: '잘' -> '자다', '갈' -> '가다')
                # 부사로 오진했더라도 '잘'은 기본 용언 원형을 '자다'로 매핑하여 힌트를 생성합니다.
                lemma = t1.form
                if original_word == "잘":
                    lemma = "자다"
                
                # 원문 글자는 100% 보존하면서, 괄호 안에 원형 힌트를 강제 주입 (범용성 핵심)
                fixed_phrase = f"{lemma} {t2.form}"
                replacements.append((target_phrase, fixed_phrase))

# 원본 문장의 손상 없이 안전하게 힌트 단어 주입 (중복 제거 후 치환)
for target, fixed in set(replacements):
    db_fetched_data = db_fetched_data.replace(target, fixed)

print(f"-> 변환된 최종 범용 원문 데이터:\n{db_fetched_data.strip()}")

-> 변환된 최종 범용 원문 데이터:
놀이터에서 처음 본 또래한테 아끼는 모래놀이 장난감 선뜻 나눠줌. 언제 이렇게 자랐나 싶어 멀리서 보는데 괜히 마음 뭉클함. 오는 길에 초코 간식 사줌.


In [21]:
refined_data = db_fetched_data.strip().replace('\n', ' ')
raw_lines = re.split(r'\.(?=\s|$)', refined_data)
lines = [line.strip() + "." for line in raw_lines if line.strip()]

step1_insights = ""
for i, line in enumerate(lines):
    step1_insights += f"{i+1}. {line}\n"
step1_insights = step1_insights.strip()

In [22]:
extract_params = {
    GenParams.DECODING_METHOD: "greedy",
    GenParams.MIN_NEW_TOKENS: 1,
    GenParams.MAX_NEW_TOKENS: 1000,
    GenParams.REPETITION_PENALTY: 1.2,
    GenParams.STOP_SEQUENCES: ["#", "Step", "주의사항"],  # 생성 흐름을 끊지 않도록 안전한 기호만 지정
}

extractor_model = ModelInference(
    model_id="mistralai/mistral-small-3-1-24b-instruct-2503",
    credentials=credentials,
    params=extract_params,
    project_id=project_id
)

In [23]:
extract_prompt = f"""[Instruction]
당신은 육아 기록 전문가입니다. 주어진 [Data]의 각 문장을 순서대로 정밀 분석하여 아래 [Output Format] 양식에 맞춰 오직 핵심 라벨 결과만 깨끗하게 출력하세요. 원문의 글자 형태를 절대로 임의로 변형하거나 깨뜨리지 마십시오.

[Data]
{step1_insights}

[Output Format]
1. 문장원문: [문장 내용]
- 핵심어: 단어1, 단어2
- 감정: 슬픔, 기쁨
- 육아범주: 수면

[Output]
"""


In [24]:
try:
    extract_response = extractor_model.generate(prompt=extract_prompt)
    if 'results' in extract_response and len(extract_response['results']) > 0:
        step2_keywords = extract_response['results'][0].get('generated_text', '').strip()
    else:
        step2_keywords = str(extract_response).strip()
except Exception as e:
    print(f"1단계 실행 중 오류 발생: {e}")
    step2_keywords = ""

perfect_match_input = step2_keywords

In [25]:
creative_params = {
    GenParams.DECODING_METHOD: "sample",  # 일기 생성 등 창의적 맥락에는 sample 방식이 자연스럽습니다.
    GenParams.MIN_NEW_TOKENS: 50,
    GenParams.MAX_NEW_TOKENS: 600,
    GenParams.REPETITION_PENALTY: 1.1,
    GenParams.TEMPERATURE: 0.1,
    GenParams.TOP_P: 0.8,
    GenParams.STOP_SEQUENCES: ["\n\n", "[END]"]
}

writer_model = ModelInference(
    model_id="meta-llama/llama-3-3-70b-instruct",
    credentials=credentials,
    params=creative_params,
    project_id=project_id
)

In [26]:
diary_prompt = f"""너는 인스타그램에서 오늘 하루의 기록을 다정하고 솔직하게 독백 형태로 공유하는 대한민국 엄마이다.
제공된 [육아 데이터 블록]의 각 번호에 명시된 '핵심어', '감정', '육아범주' 라벨 정보만을 유기적으로 조합하여, 한 번호당 정확히 한 문장씩 자연스러운 한국어로 변환해라.

[출력 예시 - 이 자연스러운 문장 연결 구조와 어투만 모방하고, 내용은 무조건 제공된 육아 데이터로만 쓰세요]
출근길에 지하철을 바로 타서 지각하지 않고 제시간에 안전하게 도착했네요.
칭찬받으려고 열심히 준비한 기획안을 부장님이 보시고 활짝 웃어주셔서 정말 뿌듯했답니다.
퇴근하고 집으로 돌아와 따뜻한 물로 샤워를 하니 하루의 피로가 싹 풀리더라고요.

[작성 규칙 - 절대 준수]
1. 문장 개수 1:1 일치: [육아 데이터 블록]의 번호 개수(현재 데이터는 3개이므로 정확히 3줄)와 똑같은 개수의 문장만 작성해라. 외부 맥락을 상상하여 문장을 추가하지 마라. [출력 예시]의 내용을 복사하지 말고, 한 문장이 끝날 때마다 무조건 줄바꿈을 해라.
2. 자연스러운 감정 및 시제 반영: 
   - '기쁨', '웃김' 같은 감정 단어를 문장 끝에 기계적으로 나열하지 마라. 오늘 실제로 겪은 과거의 일을 회상하듯 자연스러운 행동 묘사 속에 녹여내라.
   - 데이터의 핵심어를 문맥에 맞게 서술형으로 변형하되, '행복해져서 더라고요' 같은 오타나 비문이 절대 생기지 않도록 문장 끝맺음을 완벽한 한국어 문법으로 작성해라.
3. 주어 전면 생략: 문장 시작할 때 '우리 아기가~', '엄마는~' 같은 주어는 절대 쓰지 마라. 문장 첫 단어는 행동이나 상황으로 자연스럽게 시작해라.
4. 어미 결합 규칙 (필수 준수): 문장의 마지막 어미는 반드시 `~네요.`, `~했답니다.`, `~나와요.`, `~지요.` 중 하나로 결합하여 끝마쳐라. 어미 앞에 어색한 연결 조사(예: ~해서 더라고요, ~해져서 더라고요)를 절대 붙이지 마라.
5. 깨끗한 한글 출력: 문장 앞에 숫자 기호(1., 2.)나 대괄호([1번 일기])를 절대 붙이지 마라. 이모지, 기호 이모티콘, 영어나 로마자 표기는 절대로 쓰지 마라.
6. 마감 기호: 모든 문장 작성을 마친 바로 다음 줄에 무조건 [END] 라고만 출력해라.

[육아 데이터 블록]
{perfect_match_input}

[Diary]:"""


In [27]:
writer_response = writer_model.generate(prompt=diary_prompt)
writer_results = writer_response.get('results', [])
first_writer_result = next(iter(writer_results)) if isinstance(writer_results, list) and writer_results else {}
raw_diary = first_writer_result.get('generated_text', '').strip() if isinstance(first_writer_result, dict) else str(first_writer_result).strip()

In [28]:
if "[END]" in raw_diary:
    raw_diary = raw_diary.split("[END]")[0].strip()

raw_lines = [line.strip() for line in raw_diary.split('\n') if line.strip()]

full_print_lines = []
for line in raw_lines:
    line = re.sub(r'^\d+[\.\s\-~)]+|^\s*\[\d+[^\]]*\]', '', line).strip()
    
    line = re.sub(r'[\u4e00-\u9fff]', '', line)
    line = re.sub(r'[^가-힣a-zA-Z0-9\s\.,!\?\'\"~%·]', '', line).strip()
    
    if line:
        full_print_lines.append(line)

def truncate_by_bytes(text, max_bytes=400):
    text_bytes = text.encode('utf-8')
    if len(text_bytes) <= max_bytes:
        return text
    return text_bytes[:max_bytes - 3].decode('utf-8', errors='ignore').strip() + "..."

final_lines = []
for line in full_print_lines:
    final_lines.append(truncate_by_bytes(line, 400))

final_diary = "\n".join(final_lines)


In [29]:
print("\n=== 1단계: 구조화된 요약 메모 추출 완료 ===")
print(step1_insights)


=== 1단계: 구조화된 요약 메모 추출 완료 ===
1. 놀이터에서 처음 본 또래한테 아끼는 모래놀이 장난감 선뜻 나눠줌.
2. 언제 이렇게 자랐나 싶어 멀리서 보는데 괜히 마음 뭉클함.
3. 오는 길에 초코 간식 사줌.


In [30]:
print("\n=== 2단계: 주요 라벨 단어 추출 완료 ===")
print(step2_keywords)


=== 2단계: 주요 라벨 단어 추출 완료 ===
1. 문장원문: 놀이터에서 처음 본 또래한테 아끼는 모래놀이 장난감 선뜻 나눠줌.
   - 핵심어: 놀이터, 또래, 나눠줌
   - 감정: 기쁨
   - 육아범주: 사회적기능

2. 문장원문: 언제 이렇게 자랐나 싶어 멀리서 보는데 괜히 마음 뭉클함.
    - 핵심어: 자라다, 마음이 뭉클하다
    - 감정: 슬픔
    - 육아범주: 성장발달

3. 문장원문: 오는 길에 초코 간식 사줌.
     - 핵심어: 간식, 사줌
     - 감정:
     - 육아범주: 식사


In [31]:
print("\n=== 3단계: 최종 완성된 감성 일기 ===")
for idx, final_line in enumerate(final_lines):
    print(f"[{idx+1}번 일기]: {final_line}")
print(f"\n-> 최종 결과물 총 문장 수: {len(final_lines)}줄")


=== 3단계: 최종 완성된 감성 일기 ===
[1번 일기]: 놀이터에서 처음 본 또래 친구에게 아끼는 모래놀이 장난감을 선뜻 나눠줬어요.
[2번 일기]: 멀리서 바라보니까 언제 이렇게 자란 건지 생각하면서 마음이 뭉클해졌답니다.
[3번 일기]: 오늘 오는 길에 초코 간식을 사줬더니 너무 좋아했네요.

-> 최종 결과물 총 문장 수: 3줄
